# Predictive Maintenance — RUL Model (Clean Pipeline)
NASA C-MAPSS FD001 — XGBoost Regressor

This notebook is a single linear pipeline: one load, one cleaning pass,
one leakage-safe validation scheme, one ablation test, one final model.
No duplicate cells, no re-imports, no dead branches.

If `../data/train_FD001.txt` etc. are not found, a synthetic C-MAPSS-style
fallback is generated so the notebook still runs end-to-end — swap in your
real files and nothing else changes.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

RANDOM_STATE = 42
RUL_CLIP = 125  # standard C-MAPSS piecewise RUL ceiling
np.random.seed(RANDOM_STATE)


## 1. Load data

In [ ]:
DATA_DIR = "../data"

COLS = (
    ["engine_id", "cycle", "op1", "op2", "op3"]
    + [f"sensor{i}" for i in range(1, 22)]
)

def _load_real():
    train = pd.read_csv(f"{DATA_DIR}/train_FD001.txt", sep=r"\s+", header=None)
    test = pd.read_csv(f"{DATA_DIR}/test_FD001.txt", sep=r"\s+", header=None)
    rul_test = pd.read_csv(f"{DATA_DIR}/RUL_FD001.txt", sep=r"\s+", header=None)
    train.columns = COLS
    test.columns = COLS
    rul_test.columns = ["true_RUL"]
    return train, test, rul_test

def _generate_synthetic(n_engines=100):
    """Fallback only — used if real C-MAPSS files are not present."""
    rng = np.random.default_rng(RANDOM_STATE)
    rows = []
    for eid in range(1, n_engines + 1):
        life = rng.integers(120, 260)
        base = rng.normal(0, 1, size=21)
        for cyc in range(1, life + 1):
            deg = cyc / life
            noise = rng.normal(0, 0.05, size=21)
            sensors = base + deg * rng.uniform(0.3, 2.0, size=21) + noise
            row = [eid, cyc, rng.normal(), rng.normal(), rng.normal()] + list(sensors)
            rows.append(row)
    train = pd.DataFrame(rows, columns=COLS)
    return train, None, None

try:
    train_df, test_df, rul_test_df = _load_real()
    print(f"Loaded real C-MAPSS FD001 files from {DATA_DIR}/")
except FileNotFoundError:
    train_df, test_df, rul_test_df = _generate_synthetic()
    print("Real C-MAPSS files not found — using synthetic fallback for this run. "
          "Point DATA_DIR at your real files to use them.")

print(train_df.shape)
train_df.head()


## 2. Compute RUL (clipped) and drop constant / zero-variance columns

In [ ]:
max_cycle = (
    train_df.groupby("engine_id")["cycle"]
    .max()
    .reset_index()
    .rename(columns={"cycle": "max_cycle"})
)

train_df = train_df.merge(max_cycle, on="engine_id", how="left")
train_df["RUL_raw"] = train_df["max_cycle"] - train_df["cycle"]

# Piecewise/clipped RUL: early cycles look ~identical regardless of eventual
# life, so an unclipped linear label forces the model to fit noise in the
# healthy region. Clipping at RUL_CLIP is the standard C-MAPSS treatment.
train_df["RUL"] = train_df["RUL_raw"].clip(upper=RUL_CLIP)

constant_cols = train_df.columns[train_df.nunique() == 1].tolist()
print("Dropping constant columns:", constant_cols)
train_df = train_df.drop(columns=constant_cols)


## 3. EDA (kept minimal — one plot per question)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(train_df["RUL"], bins=30, edgecolor="black")
axes[0].set_title("RUL distribution (clipped)")
axes[0].set_xlabel("RUL (cycles)")

engine_life = train_df.groupby("engine_id")["cycle"].max()
axes[1].hist(engine_life, bins=20, edgecolor="black")
axes[1].set_title("Engine life distribution")
axes[1].set_xlabel("Max cycle")

eng1 = train_df[train_df["engine_id"] == 1]
axes[2].plot(eng1["cycle"], eng1["sensor11"])
axes[2].set_title("Sensor11 — Engine 1")
axes[2].set_xlabel("Cycle")

plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 10))
sns.heatmap(train_df.drop(columns=["max_cycle", "RUL_raw"]).corr(), cmap="coolwarm", center=0)
plt.title("Feature correlation heatmap")
plt.show()


## 4. Feature set + leakage-safe split

`GroupShuffleSplit` on `engine_id` — no engine appears in both train and test.
This replaces the earlier row-wise `train_test_split`, which let cycles from
the same engine leak across the split.

In [ ]:
FEATURE_COLS = [c for c in train_df.columns if c not in
                 ("RUL", "RUL_raw", "max_cycle", "engine_id")]

X = train_df[FEATURE_COLS]
y = train_df["RUL"]
groups = train_df["engine_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"Engine overlap between train/test: {len(overlap)}  (must be 0)")
assert len(overlap) == 0


## 5. GroupKFold cross-validation (honest metric, not a single split)

In [ ]:
def run_group_kfold(X, y, groups, n_splits=5, **xgb_params):
    gkf = GroupKFold(n_splits=n_splits)
    maes, r2s = [], []
    for tr_idx, te_idx in gkf.split(X, y, groups):
        model = XGBRegressor(random_state=RANDOM_STATE, **xgb_params)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        pred = model.predict(X.iloc[te_idx])
        maes.append(mean_absolute_error(y.iloc[te_idx], pred))
        r2s.append(r2_score(y.iloc[te_idx], pred))
    return np.array(maes), np.array(r2s)

XGB_PARAMS = dict(n_estimators=300, max_depth=6, learning_rate=0.05)

maes, r2s = run_group_kfold(X, y, groups, n_splits=5, **XGB_PARAMS)

print(f"Full-feature model  —  MAE: {maes.mean():.2f} ± {maes.std():.2f}   "
      f"R²: {r2s.mean():.4f} ± {r2s.std():.4f}")


## 6. Ablation test — is the model just counting cycles?

Same GroupKFold procedure, `cycle` removed from the feature set. If R² holds
up reasonably well without it, the sensors are doing real work.

In [ ]:
ABLATION_COLS = [c for c in FEATURE_COLS if c != "cycle"]
X_ablation = train_df[ABLATION_COLS]

maes_abl, r2s_abl = run_group_kfold(X_ablation, y, groups, n_splits=5, **XGB_PARAMS)

print(f"Sensors-only model   —  MAE: {maes_abl.mean():.2f} ± {maes_abl.std():.2f}   "
      f"R²: {r2s_abl.mean():.4f} ± {r2s_abl.std():.4f}")
print()
retained = r2s_abl.mean() / r2s.mean() * 100
print(f"Sensors-only model retains {retained:.1f}% of full-model R².")


## 7. Final model — fit once on the full training split, used everywhere downstream

In [ ]:
final_model = XGBRegressor(random_state=RANDOM_STATE, **XGB_PARAMS)
final_model.fit(X_train, y_train)

pred_test = final_model.predict(X_test)
print(f"Held-out split  —  MAE: {mean_absolute_error(y_test, pred_test):.2f}   "
      f"R²: {r2_score(y_test, pred_test):.4f}")


## 8. Feature importance + diagnostic plots

In [ ]:
importance = (
    pd.DataFrame({"Feature": FEATURE_COLS, "Importance": final_model.feature_importances_})
    .sort_values("Importance", ascending=False)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].barh(importance["Feature"][:10][::-1], importance["Importance"][:10][::-1])
axes[0].set_title("Top 10 feature importance")

axes[1].scatter(y_test, pred_test, alpha=0.4)
axes[1].plot([0, RUL_CLIP], [0, RUL_CLIP], "r--")
axes[1].set_title("Actual vs predicted RUL")
axes[1].set_xlabel("Actual"); axes[1].set_ylabel("Predicted")

residuals = y_test.values - pred_test
axes[2].scatter(pred_test, residuals, alpha=0.4)
axes[2].axhline(0, color="red", linestyle="--")
axes[2].set_title("Residuals")
axes[2].set_xlabel("Predicted RUL"); axes[2].set_ylabel("Residual")

plt.tight_layout()
plt.show()

importance.head(10)


## 9. SHAP explainability

In [ ]:
import shap

explainer = shap.TreeExplainer(final_model)
sample = X_test.sample(min(300, len(X_test)), random_state=RANDOM_STATE)
shap_values = explainer(sample)

shap.plots.beeswarm(shap_values, max_display=10)


## 10. Digital-twin snapshot (seeded — reproducible across reruns)

One snapshot cycle per engine, sampled from the 60-90% point of its life,
so engines look "currently in service" instead of "one cycle from failure".
Seeded with `RANDOM_STATE` so the same engines get the same snapshot cycle
every time this cell runs.

In [ ]:
snap_rng = np.random.default_rng(RANDOM_STATE)

snapshot_rows = []
for eng in train_df["engine_id"].unique():
    eng_df = train_df[train_df["engine_id"] == eng]
    life = eng_df["cycle"].max()
    lo, hi = max(1, int(0.6 * life)), max(2, int(0.9 * life))
    snap_cycle = snap_rng.integers(lo, hi)
    row = eng_df[eng_df["cycle"] == snap_cycle]
    if len(row):
        snapshot_rows.append(row)

snapshot_df = pd.concat(snapshot_rows).reset_index(drop=True)
snapshot_df["Predicted_RUL"] = final_model.predict(snapshot_df[FEATURE_COLS])

snapshot_df[["engine_id", "cycle", "RUL", "Predicted_RUL"]].head()


## 11. Health index, failure risk, maintenance recommendation

In [ ]:
snapshot_df["Health_Index"] = (snapshot_df["Predicted_RUL"] / RUL_CLIP).clip(0, 1) * 100
snapshot_df["Failure_Risk_%"] = 100 - snapshot_df["Health_Index"]

def recommendation(risk_pct):
    if risk_pct >= 80:
        return "Immediate Maintenance"
    elif risk_pct >= 50:
        return "Schedule Maintenance"
    return "Healthy"

snapshot_df["Recommendation"] = snapshot_df["Failure_Risk_%"].apply(recommendation)

print(snapshot_df["Failure_Risk_%"].describe())
snapshot_df[["engine_id", "cycle", "Predicted_RUL", "Health_Index",
             "Failure_Risk_%", "Recommendation"]].head(10)


## 12. Export for the Streamlit dashboard

Persist the fitted model and the snapshot table so the dashboard loads
them instead of retraining / re-randomizing on every run.

In [ ]:
import joblib

# Model
joblib.dump(
    final_model,
    "xgb_rul_model.pkl"
)

# Dashboard datasets
snapshot_df.to_csv(
    "digital_twin_snapshot.csv",
    index=False
)

train_df.to_csv(
    "train_processed.csv",
    index=False
)

test_last[
    [
        'engine_id',
        'Actual_RUL',
        'Predicted_RUL',
        'Actual_RUL_clipped'
    ]
].to_csv(
    "test_predictions.csv",
    index=False
)

importance.to_csv(
    "feature_importance.csv",
    index=False
)

print("All dashboard files saved.")

In [ ]:
import os

print("Current Working Directory:")
print(os.getcwd())

In [ ]:
import os

print(os.path.exists("../data"))
print(os.path.exists("data"))

In [ ]:
import os

print(os.listdir("../data"))

In [ ]:
import os

print(os.path.abspath("../data"))

In [ ]:
print(test_df.head())
print(test_df.shape)

In [ ]:
print(rul_test_df.head())
print(rul_test_df.shape)

In [ ]:
test_last = (
    test_df
    .groupby('engine_id')
    .tail(1)
    .copy()
)

test_last.head()

In [ ]:
test_last.shape

In [ ]:
test_last = test_last.reset_index(drop=True)

test_last['Actual_RUL'] = (
    rul_test_df['true_RUL']
)

In [ ]:
test_last[
    ['engine_id','Actual_RUL']
].head()

In [ ]:
FEATURE_COLS = [
    'cycle',
    'op1','op2',
    'sensor2','sensor3','sensor4',
    'sensor6','sensor7','sensor8',
    'sensor9','sensor11','sensor12',
    'sensor13','sensor14','sensor15',
    'sensor17','sensor20','sensor21'
]

In [ ]:
X_official = test_last[
    FEATURE_COLS
]

In [ ]:
globals().keys()

In [ ]:
%who

In [ ]:
official_pred = final_model.predict(
    X_official
)

In [ ]:
test_last['Predicted_RUL'] = official_pred

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    test_last['Actual_RUL'],
    test_last['Predicted_RUL']
)

rmse = np.sqrt(
    mean_squared_error(
        test_last['Actual_RUL'],
        test_last['Predicted_RUL']
    )
)

r2 = r2_score(
    test_last['Actual_RUL'],
    test_last['Predicted_RUL']
)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

In [ ]:
print(X_official.columns)
print(FEATURE_COLS)

In [ ]:
(rul_test_df['true_RUL'] > 125).sum()

In [ ]:
test_last['Actual_RUL_clipped'] = (
    test_last['Actual_RUL']
    .clip(upper=125)
)

In [ ]:
mae_clip = mean_absolute_error(
    test_last['Actual_RUL_clipped'],
    test_last['Predicted_RUL']
)

rmse_clip = np.sqrt(
    mean_squared_error(
        test_last['Actual_RUL_clipped'],
        test_last['Predicted_RUL']
    )
)

r2_clip = r2_score(
    test_last['Actual_RUL_clipped'],
    test_last['Predicted_RUL']
)

print("Clipped MAE :", mae_clip)
print("Clipped RMSE:", rmse_clip)
print("Clipped R²  :", r2_clip)

In [ ]:
print(test_last.shape)
print(test_last['engine_id'].nunique())

In [ ]:
test_last[['engine_id','cycle']].head()

In [ ]:
set(FEATURE_COLS) - set(X_official.columns)

In [ ]:
set(X_official.columns) - set(FEATURE_COLS)

In [ ]:
%who DataFrame

In [ ]:
print(test_last.columns)

In [ ]:
test_last[
    [
        'engine_id',
        'Actual_RUL',
        'Predicted_RUL',
        'Actual_RUL_clipped'
    ]
].to_csv(
    "test_predictions.csv",
    index=False
)